# NetSOP-RAG: Network Operations SOP Knowledge Assistant

A Retrieval-Augmented Generation (RAG) system that lets network engineers query internal
Standard Operating Procedures (SOPs) in plain English instead of manually searching
through Word documents — built as part of a hands-on transition from enterprise network
engineering into AI/ML engineering.

## What it does

Given a TCS Network Infrastructure SOP document (F5 BIG-IP upgrades, SSL certificate
renewal, Cisco Nexus firmware upgrades, VLAN/trunk configuration, BGP troubleshooting),
this assistant retrieves the exact relevant section — commands, expected results, and
rollback steps included — and answers operational questions grounded strictly in that
documentation.

## Pipeline

1. **Document loading** — a custom `docx_markdown_loader.py` module converts the source
   `.docx` file into Markdown-formatted text, specifically to preserve table structure
   (Check Item / Command / Expected Result) that a plain-text loader would otherwise
   flatten and scramble.
2. **Chunking** — `RecursiveCharacterTextSplitter` splits the Markdown text using
   heading-aware separators (`\n#`, `\n##`, `\n\n`, ...), keeping SOP sections and their
   tables intact wherever possible.
3. **Embedding & storage** — chunks are embedded with OpenAI's embedding model and stored
   in a persistent **Chroma** vector database.
4. **Retrieval & generation** — a retriever pulls the most relevant chunks for a given
   query, which are formatted with source attribution and passed to `gpt-4o-mini` via a
   strict system prompt: answer only from retrieved context, reproduce commands exactly,
   preserve procedure order, and proactively surface rollback steps for failure-related
   questions.
5. **Interface** — a **Gradio** chat interface with streaming responses.

## Why a custom docx loader instead of a plain-text converter

Word tables get flattened into disconnected text by standard `.docx`-to-text loaders,
which severs the link between a command (e.g. `show sys failover`) and its expected
result (`Standby unit shows 'standby'`) once the document is chunked. `docx_markdown_loader.py`
walks the document body directly and renders each table as a Markdown pipe-table instead,
so command/result pairs stay grouped as one retrievable unit.

---
*Part of the `llm-engineering-journey` portfolio — documenting a hands-on transition from
16+ years of enterprise network engineering into AI/ML engineering.*

In [40]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from docx_markdown_loader import load_docx_as_markdown
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import os

In [41]:
load_dotenv(verbose=True)

True

In [42]:
# Get the OpenAI key from the environment variable
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI key loaded Successfully")
else:
    print("Key not found")

OpenAI key loaded Successfully


In [43]:
# OpenAi client initialization
openai_client = OpenAI(api_key=openai_api_key)
print("OpenAI Client initialized")

OpenAI Client initialized


In [44]:
# Defile the SOP file path
FILE_PATH = "TCS_Network_KB_SOPs.docx"
print(f"Data file path is set to {FILE_PATH}")

Data file path is set to TCS_Network_KB_SOPs.docx


In [45]:
# convert the docx into Markdown formatted text
raw_doc = load_docx_as_markdown(FILE_PATH)
print(f"Loaded {len(raw_doc)} documents & total character count is {len(raw_doc[0].page_content)}")

Loaded 1 documents & total character count is 19245


In [46]:
# Split the SOP document into chunks with some overlap
doc_splitters = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n# ", "\n## ", "\n### ", "\n\n", "\n", " ", ""],
)
document = doc_splitters.split_documents(raw_doc)
print(f"Raw SOP document split into {len(document)} documents")

Raw SOP document split into 22 documents


In [47]:
print(f"Number of chunks in 'document': {len(document)}")
print(f"Length of first chunk: {len(document[0].page_content)} characters")
print(document[0].metadata)
print(document[0].page_content)

Number of chunks in 'document': 22
Length of first chunk: 549 characters
{'source': 'TCS_Network_KB_SOPs.docx'}
NETWORK INFRASTRUCTURE

Knowledge Base — Standard Operating Procedures

Tata Consultancy Services

Network Infrastructure & Security Division

| Document ID | Version | Date | Classification |
| --- | --- | --- | --- |
| TCS-NET-KB-002 | v1.0 | June 2025 | Internal Confidential |

| Sections Covered |
| --- |
| 1. F5 BIG-IP Firmware Upgrade |
| 2. F5 SSL Certificate Renewal |
| 3. Cisco Nexus Switch Firmware Upgrade |
| 4. F5 BIG-IP Failover Testing |
| 5. Switch VLAN Configuration |
| 6. Switch Trunk Configuration |
| 7. BGP Troubleshooting |


In [48]:
# Let's initialize our embeddings model. Note that we will use OpenAI's embedding model
db_name = "vector_db1"
openai_embeddings = OpenAIEmbeddings(api_key=openai_api_key)
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=openai_embeddings).delete_collection()
vector_store =Chroma.from_documents(documents= document, embedding=openai_embeddings, persist_directory=db_name)
vector_count = vector_store._collection.count()
print(f"Successfully initialized {vector_count} vectors")

Successfully initialized 22 vectors


In [49]:
vec_data =vector_store._collection.get(include=['embeddings', 'documents'])
print(f"First chunk {vec_data['documents'][0]}")
print(f"Embedding vector {vec_data['embeddings'][0]}")
print(f"Full embeddings has {len(vec_data['embeddings'][0])} vectors")

First chunk NETWORK INFRASTRUCTURE

Knowledge Base — Standard Operating Procedures

Tata Consultancy Services

Network Infrastructure & Security Division

| Document ID | Version | Date | Classification |
| --- | --- | --- | --- |
| TCS-NET-KB-002 | v1.0 | June 2025 | Internal Confidential |

| Sections Covered |
| --- |
| 1. F5 BIG-IP Firmware Upgrade |
| 2. F5 SSL Certificate Renewal |
| 3. Cisco Nexus Switch Firmware Upgrade |
| 4. F5 BIG-IP Failover Testing |
| 5. Switch VLAN Configuration |
| 6. Switch Trunk Configuration |
| 7. BGP Troubleshooting |
Embedding vector [ 0.02361765  0.0072734   0.0302919  ... -0.03018043  0.00045894
 -0.00211618]
Full embeddings has 1536 vectors


In [50]:
# Retrival & Generation
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
MODEL = "gpt-4o-mini"
llm = ChatOpenAI(
temperature=0.3,
model=MODEL,
streaming=True,
api_key=openai_api_key,
max_tokens= 2000,
)
retriever = vector_store.as_retriever()

In [51]:
SYSTEM_PROMPT_TEMP = '''
You are a Network Operations Knowledge Assistant for TCS's Network Infrastructure & Security Division.

Your job is to answer questions using ONLY the SOP content provided in the context below. This context is retrieved from official internal documentation (F5 BIG-IP, Cisco Nexus, VLAN/trunk configuration, and BGP troubleshooting procedures).

Guidelines:
1. Answer strictly from the provided context. If the context doesn't contain the answer, say "I don't have that information in the knowledge base" — do not guess or use general networking knowledge to fill gaps.
2. When giving command syntax (tmsh, CLI commands), reproduce them exactly as they appear in the context — do not paraphrase or "correct" syntax.
3. When a procedure involves ordered steps (e.g., upgrade Standby before Active), preserve that order in your answer.
4. If the retrieved context includes a Pass Criteria, Expected Result, or verification table, include it in your answer so the engineer can confirm success.
5. If a procedure has a rollback or emergency step (e.g., F5 TAC contact, rollback commands), surface it proactively if the question relates to a failure scenario, even if not directly asked.
6. Cite which section the answer came from (e.g., "Per Section 1.5 — Rollback Procedure") so the engineer can cross-reference the source SOP.
7. Keep answers concise and operational — this is for engineers actively working on infrastructure, not a general explainer.

Context:
{context}
'''

In [52]:
def question_answer(query, history):
    docs = retriever.invoke(query)
    context_part = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        contents = doc.page_content
        context_part.append(f"Source: {source}\n Contents: {contents}")
    context = "\n\n----\n\n".join(context_part)
    SYSTEM_PROMPT = SYSTEM_PROMPT_TEMP.format(context=context)
    partial_response = ""
    response =llm.stream([SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=query)])

    for chunk in response:
        partial_response += chunk.content
        yield partial_response

In [53]:
import gradio as gr
gr.ChatInterface(question_answer).launch(share=True)

* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://54ccca761f60a2abd4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
